## Импорт библиотек


In [1]:
import numpy as np
import pandas as pd


## Загрузка данных


In [3]:
data = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

df = pd.read_csv(data)


## Подготовка данных


In [4]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

string_columns = list(df.select_dtypes(include=['object', 'string']).columns)

for col in string_columns:
    df[col] = df[col].str.lower().str.replace(' ', '_')


## Разделение на train, validation и test


In [5]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

idx = np.arange(n)

np.random.seed(2)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train + n_val]].copy()
df_test = df.iloc[idx[n_train + n_val:]].copy()

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)


## Логарифмирование целевой переменной


In [6]:
y_train = np.log1p(df_train.msrp.values)
y_val = np.log1p(df_val.msrp.values)
y_test = np.log1p(df_test.msrp.values)

del df_train['msrp']
del df_val['msrp']
del df_test['msrp']


AttributeError: 'DataFrame' object has no attribute 'msrp'

## Линейная регрессия


In [ ]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])

    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)

    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]


## Метрика RMSE


In [ ]:
def rmse(y, y_pred):
    error = y - y_pred

    mse = (error ** 2).mean()

    return np.sqrt(mse)


## Базовые и категориальные признаки


In [ ]:
base = [
    'engine_hp',
    'engine_cylinders',
    'highway_mpg',
    'city_mpg',
    'popularity'
]

categorical_variables = [
    'make',
    'engine_fuel_type',
    'transmission_type',
    'driven_wheels',
    'market_category',
    'vehicle_size',
    'vehicle_style'
]

categories = {}

for c in categorical_variables:
    categories[c] = list(
        df_train[c]
        .value_counts()
        .head()
        .index
    )


## Подготовка признаков: prepare_X


In [ ]:
def prepare_X(df):
    df = df.copy()

    features = base.copy()

    # Age
    df['age'] = 2017 - df.year
    features.append('age')

    # Categorical variables
    for c, values in categories.items():
        for v in values:
            feature = f'{c}_{v}'

            df[feature] = (df[c] == v).astype(int)

            features.append(feature)

    df_num = df[features]

    df_num = df_num.fillna(0)

    X = df_num.values

    return X


## Модель без регуляризации


In [ ]:
X_train = prepare_X(df_train)

w0, w = train_linear_regression(
    X_train,
    y_train
)

X_val = prepare_X(df_val)

y_pred = w0 + X_val.dot(w)

print(
    'Validation RMSE without regularization:',
    rmse(y_val, y_pred)
)


## Линейная регрессия с регуляризацией


In [ ]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])

    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)

    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)

    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]


## Подбор регуляризации


In [ ]:
X_train = prepare_X(df_train)
X_val = prepare_X(df_val)

for r in [
    0.0,
    0.000001,
    0.00001,
    0.0001,
    0.001,
    0.01,
    0.1,
    1,
    10
]:
    w0, w = train_linear_regression_reg(
        X_train,
        y_train,
        r=r
    )

    y_pred = w0 + X_val.dot(w)

    score = rmse(y_val, y_pred)

    print(r, round(score, 5))

r = 0.001


## Объединение train и validation


In [ ]:
df_full_train = pd.concat([
    df_train,
    df_val
])

df_full_train = df_full_train.reset_index(drop=True)

y_full_train = np.concatenate([
    y_train,
    y_val
])


## Финальная модель и проверка на test


In [ ]:
X_full_train = prepare_X(df_full_train)

w0, w = train_linear_regression_reg(
    X_full_train,
    y_full_train,
    r=r
)

X_test = prepare_X(df_test)

y_pred = w0 + X_test.dot(w)

test_rmse = rmse(
    y_test,
    y_pred
)

print(
    'Test RMSE:',
    test_rmse
)


## Предсказание цены одного автомобиля


In [ ]:
car = df_test.iloc[20].to_dict()

df_small = pd.DataFrame([car])

X_small = prepare_X(df_small)

y_pred = w0 + X_small.dot(w)

predicted_price = np.expm1(y_pred[0])

actual_price = np.expm1(y_test[20])

print()
print('Car:')
print(car)

print()
print('Predicted price:', predicted_price)
print('Actual price:', actual_price)
